# Week 9 Live Coding
## The donor's portfolio

Four GOTV tactics. Each has a published effect and a cost. Which produces the most votes per dollar?

Four things we will do:
1. Compute cost per vote for each tactic
2. Compute total votes produced per \$500K
3. Show how uncertainty changes the ranking
4. Update with a new canvassing study and re-rank

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below to load the data.

In [ ]:
import pandas as pd
import numpy as np

tactics = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/'
                      'data_science_campaigns_26/main/weeks/wk09_cost_effectiveness/data/'
                      'gotv_tactics.csv')
tactics

## Part 1 — Cost per additional vote

The formula is the same one we've been using since Week 3:

**cost per vote = cost per contact / (effect size as a proportion)**

At +2.0 pp and \$20/conversation: \$20 / 0.02 = \$1,000 per vote.

In [ ]:
# Cost per additional vote for each tactic
tactics['cost_per_vote'] = tactics['cost_per_contact'] / (tactics['effect_pp'] / 100)
tactics[['tactic', 'effect_pp', 'cost_per_contact', 'cost_per_vote']]

**Stop and read the `cost_per_vote` column.** The ranking by effect (canvassing > mail > phone > texts) is NOT the same as the ranking by cost-effectiveness (texts ≈ mail > phone > canvassing).

The best tactic *per person* is not the best tactic *per dollar*.

## Part 2 — Total votes per \$500K

How many additional votes does each tactic produce with a \$500K budget?

In [ ]:
budget = 500_000

# How many contacts can you afford?
tactics['contacts'] = (budget / tactics['cost_per_contact']).astype(int)

# How many additional votes does that produce?
tactics['additional_votes'] = tactics['contacts'] * (tactics['effect_pp'] / 100)

tactics[['tactic', 'cost_per_contact', 'contacts', 'effect_pp', 'additional_votes']].round(0)

Mail produces far more total votes than canvassing for the same \$500K — because you reach vastly more people, even though the per-person effect is smaller.

This is the W1 lesson at work: **the low marginal cost of mail lets you scale**. Canvassing has a high marginal cost, which limits your reach.

## Part 3 — Uncertainty changes the ranking

Every effect estimate has a confidence interval. Let's see what happens to cost-per-vote at the edges of the CI.

In [ ]:
# Cost per vote at the low and high end of the CI
# Note the inverse: a HIGHER effect means LOWER cost per vote.
# So cpv_low_ci uses effect_high_pp, and vice versa.
tactics['cpv_low_ci'] = tactics['cost_per_contact'] / (tactics['effect_high_pp'] / 100)
tactics['cpv_high_ci'] = tactics['cost_per_contact'] / (tactics['effect_low_pp'] / 100)

# Handle negative or zero lower bounds (texts CI includes 0)
tactics.loc[tactics['effect_low_pp'] <= 0, 'cpv_high_ci'] = float('inf')

print('Cost per vote with uncertainty:')
for _, row in tactics.iterrows():
    high_str = f'\${row["cpv_high_ci"]:,.0f}' if row['cpv_high_ci'] != float('inf') else 'infinite (CI includes 0)'
    print(f'  {row["tactic"]:25s}  \${row["cpv_low_ci"]:>7,.0f}  to  {high_str}')

Texts look great at the point estimate (\$80/vote) but the CI includes zero. If the true effect is zero, the cost per vote is infinite — you're spending money for nothing.

**Uncertainty matters for the ranking.** A donor who is risk-averse might prefer mail (narrower CI, lower floor) over texts (wider CI, higher ceiling but also higher risk).

## Part 4 — A new canvassing study arrives

A 2024 field experiment finds canvassing effects of +0.8 pp (CI: [+0.2, +1.4]) at \$25/conversation. How does this change the ranking?

In [ ]:
new_study = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/'
                        'data_science_campaigns_26/main/weeks/wk09_cost_effectiveness/data/'
                        'new_canvassing_study.csv')
print(new_study.to_string(index=False))

In [ ]:
# Cost per vote under the new study
new_cpv = 25.00 / (0.8 / 100)
print(f'New canvassing cost per vote: \${new_cpv:,.0f}')
print(f'Old canvassing cost per vote: \$1,000')
print(f'\nCanvassing went from \$1,000/vote to \${new_cpv:,.0f}/vote.')
print(f'That is a {new_cpv/1000:.0f}x increase.')

Under the new study, canvassing costs \$3,125 per vote — more than 3× the old estimate. The new study has a smaller effect (+0.8 vs. +2.0 pp) AND a higher cost (\$25 vs. \$20).

**But should you use the new study alone?** The meta-analysis is based on dozens of experiments. The new study is one experiment. The new study is in the *right* setting, but one study shouldn't erase everything that came before.

**The Bayesian intuition:** your updated belief should be *between* the old number (+2.0 pp) and the new number (+0.8 pp). Maybe +1.2 or +1.5 pp. The exact number depends on how much weight you give the new study vs. the prior evidence.

In [ ]:
# What's the cost per vote at an updated belief of +1.2 pp?
# These numbers are illustrative — in practice, a formal meta-analysis would
# compute the weighted average. The point: the answer is between the two, not either extreme.
updated_effect = 1.2  # pp — a compromise between +2.0 (prior) and +0.8 (new study)
updated_cost = 22.50  # split the difference on cost too
updated_cpv = updated_cost / (updated_effect / 100)

print(f'Updated canvassing estimate:')
print(f'  Effect: +{updated_effect} pp')
print(f'  Cost: \${updated_cost}/conversation')
print(f'  Cost per vote: \${updated_cpv:,.0f}')
print(f'\nCompare: mail is still \$86/vote. Canvassing at \${updated_cpv:,.0f}/vote is {updated_cpv/86:.0f}x more expensive.')

---

## What you've seen today

- **Cost per vote = cost per contact / effect size.** The ranking by effect is not the ranking by cost-effectiveness.
- **Uncertainty matters.** A tactic with a CI that includes zero has an infinite worst-case cost per vote.
- **One new study should move your belief, not replace it.** Bayesian updating: prior + evidence → updated belief.
- **Fixed costs and diminishing returns** mean the optimal allocation is usually a mix, not all-in on one tactic.

**Lying-with-data tag #8:** Comparing by effect size without cost. Ignoring uncertainty in cost-per-vote.

Next, open `problem_set.ipynb`.